# Slide 12 demo: PII detection and six privacy actions

**Architecture:** VS Code notebook → AWS Lambda → Amazon Comprehend `DetectPiiEntities`. KMS encrypts text and token-vault values. S3 stores encrypted token-vault entries and quarantined text.

| Action | Result | Where original text goes |
|---|---|---|
| Mask | Detected spans become `*` | Not stored |
| Replace | Detected spans become `[EMAIL]`, `[PHONE]`, etc. | Not stored |
| Tokenize | Each span becomes a random token | Vault entry is KMS encrypted and kept in S3 |
| Encrypt | Whole input becomes KMS ciphertext | Ciphertext is returned; no raw text in response |
| Quarantine | Return only a reference | Full input is stored under a restricted S3 prefix with SSE-KMS |
| Reject | Return a rejection decision | Not stored |

**Comprehend detects PII; Lambda implements the six policies.** Comprehend's synchronous API returns spans, labels, and confidence. For this notebook, use synthetic English text only. Its API supports up to 100 KB, while direct KMS Encrypt has a smaller plaintext limit, so this demonstration caps inputs at 3,000 UTF-8 bytes.

**Before class in one AWS Region:**

1. Install VS Code Python and Jupyter extensions. In the terminal, create a virtual environment, activate it, and run `python -m pip install boto3 ipykernel`. On PowerShell: `python -m venv .venv` then `.venv\Scripts\Activate.ps1`. Select the `.venv` kernel.
2. Configure an AWS profile, ideally with SSO: `aws sso login --profile training`. The notebook uses this profile to create and invoke Lambda. The notebook principal needs `lambda:CreateFunction`, `lambda:GetFunction`, `lambda:InvokeFunction`, `lambda:DeleteFunction`, `iam:PassRole` for the supplied role, and `sts:GetCallerIdentity`.
3. Create an **existing** S3 bucket in that Region and an **existing** customer-managed symmetric KMS key. Use a dedicated training bucket; block public access. Supply their names below.
4. Create an IAM role trusted by `lambda.amazonaws.com`. Attach `AWSLambdaBasicExecutionRole` for logs and grant the role `comprehend:DetectPiiEntities`, `kms:Encrypt` and `kms:GenerateDataKey` on the chosen key, and `s3:PutObject` on `arn:aws:s3:::YOUR-BUCKET/pii-demo/*`. The KMS key policy must allow the role. S3 SSE-KMS writes require data-key permission. The notebook principal only needs permission to pass this role, not to see protected vault objects.
5. Set the configuration cell and run the notebook **one cell at a time**. Creating the Lambda and calling AWS services can incur charges. The last cell deletes the demo function; S3 vault and quarantine objects remain until you delete them under your retention policy.

**Example inline policy for the Lambda role** (replace the bucket and key ARN; scope Comprehend as required by its IAM action):

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {"Effect": "Allow", "Action": "comprehend:DetectPiiEntities", "Resource": "*"},
    {"Effect": "Allow", "Action": ["kms:Encrypt", "kms:GenerateDataKey"], "Resource": "YOUR-KMS-KEY-ARN"},
    {"Effect": "Allow", "Action": "s3:PutObject", "Resource": "arn:aws:s3:::YOUR-BUCKET/pii-demo/*"}
  ]
}
```

**Trainer script:** “Here is our synthetic record. Comprehend finds sensitive spans. Lambda applies a policy that depends on whether the application needs structure, reversibility, confidentiality, human review, or outright rejection.”


In [ ]:
import json
SAMPLE_TEXT = (
    "Customer Maya Patel wrote from maya.patel@example.com and called +1-415-555-0132. "
    "Please send the claim update to 123 Market Street, San Francisco, CA 94103."
)
print("STEP 1 — Synthetic sample data (display intentionally limited to classroom data):")
print(SAMPLE_TEXT)


## Step 2A: configure the real AWS demo

The creation switch starts off. Fill in the role, bucket, and KMS key, then set `RUN_AWS = True`. No credentials are saved in this notebook. Bucket and key must be in the same Region as Lambda. Deployment uses a unique function name so it does not overwrite another class's function.


In [ ]:
RUN_AWS = False    # Set True after creating the role, bucket and KMS key below.
AWS_PROFILE = None         # Use the default AWS credential chain on this machine.
AWS_REGION = "<AWS_REGION>"  # e.g. "ap-south-1" or "us-east-1"
LAMBDA_ROLE_ARN = "arn:aws:iam::<AWS_ACCOUNT_ID>:role/<YOUR_LAMBDA_ROLE_NAME>"
VAULT_BUCKET = "<YOUR_BUCKET_NAME>"
KMS_KEY_ARN = "arn:aws:kms:<AWS_REGION>:<AWS_ACCOUNT_ID>:key/<YOUR_KMS_KEY_ID>"

if RUN_AWS:
    import boto3
    session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
    sts = session.client("sts")
    lambda_client = session.client("lambda")
    account_id = sts.get_caller_identity()["Account"]
    if not all((LAMBDA_ROLE_ARN, VAULT_BUCKET, KMS_KEY_ARN)):
        raise ValueError("Set LAMBDA_ROLE_ARN, VAULT_BUCKET, and KMS_KEY_ARN before enabling AWS.")
    print("Account:", account_id, "Region:", AWS_REGION, "Bucket:", VAULT_BUCKET)
else:
    print("Configure AWS resources and set RUN_AWS=True to run the live Lambda demo.")

## Step 2B: inspect the Lambda handler

This is the **actual source deployed** to Lambda. It calls Comprehend for every action and never writes raw input to logs. `mask`, `replace`, and `tokenize` work from detected character offsets. It skips overlapping results, favoring the higher confidence detection. The sample is ASCII so offsets are straightforward; production code should test multilingual offsets and more complex overlap policies.


In [ ]:
LAMBDA_SOURCE = r'''
import base64
import json
import os
import uuid

import boto3

comprehend = boto3.client("comprehend")
kms = boto3.client("kms")
s3 = boto3.client("s3")
BUCKET = os.environ["VAULT_BUCKET"]
KEY = os.environ["KMS_KEY_ARN"]
ALLOWED = {"detect", "mask", "replace", "tokenize", "encrypt", "quarantine", "reject"}

def chosen_entities(text, entities):
    # Resolve overlaps by confidence, then reconstruct in source order.
    occupied = []
    selected = []
    for entity in sorted(entities, key=lambda e: -e["Score"]):
        start, end = entity["BeginOffset"], entity["EndOffset"]
        if start < 0 or end > len(text) or start >= end:
            raise ValueError("Invalid detection offsets")
        if any(start < b and end > a for a, b in occupied):
            continue
        occupied.append((start, end))
        selected.append(entity)
    return sorted(selected, key=lambda e: e["BeginOffset"])

def encrypt_bytes(value):
    return kms.encrypt(KeyId=KEY, Plaintext=value)["CiphertextBlob"]

def lambda_handler(event, context):
    action = event.get("action")
    text = event.get("text")
    if action not in ALLOWED or not isinstance(text, str) or not text:
        return {"status": "invalid_request", "message": "Supply an action and nonempty text"}
    if len(text.encode("utf-8")) > 3000:
        return {"status": "invalid_request", "message": "This demo accepts at most 3000 UTF-8 bytes"}

    raw = comprehend.detect_pii_entities(Text=text, LanguageCode="en")["Entities"]
    entities = chosen_entities(text, raw)
    summary = [{"type": e["Type"], "confidence": round(e["Score"], 3)} for e in entities]
    base = {"action": action, "entity_count": len(entities), "detected": summary}
    if action == "detect":
        return {**base, "status": "detected"}
    if not entities:
        # A detection miss must not cause the handler to echo raw input.
        return {**base, "status": "review_required", "message": "No PII spans identified"}
    if action == "reject":
        return {**base, "status": "rejected", "reason": "PII detected"}
    if action == "quarantine":
        object_key = "pii-demo/quarantine/" + str(uuid.uuid4()) + ".txt"
        s3.put_object(Bucket=BUCKET, Key=object_key, Body=text.encode("utf-8"),
                      ContentType="text/plain", ServerSideEncryption="aws:kms", SSEKMSKeyId=KEY)
        return {**base, "status": "quarantined", "s3_uri": f"s3://{BUCKET}/{object_key}"}
    if action == "encrypt":
        ciphertext = encrypt_bytes(text.encode("utf-8"))
        return {**base, "status": "encrypted", "ciphertext_base64": base64.b64encode(ciphertext).decode("ascii")}

    segments = []
    cursor = 0
    for e in entities:
        start, end = e["BeginOffset"], e["EndOffset"]
        segments.append(text[cursor:start])
        original = text[start:end]
        if action == "mask":
            substitute = "*" * len(original)
        elif action == "replace":
            substitute = "[" + e["Type"] + "]"
        else:  # tokenize
            token = str(uuid.uuid4())
            ciphertext = encrypt_bytes(original.encode("utf-8"))
            s3.put_object(Bucket=BUCKET, Key="pii-demo/vault/" + token + ".bin",
                          Body=ciphertext, ContentType="application/octet-stream",
                          ServerSideEncryption="aws:kms", SSEKMSKeyId=KEY)
            substitute = "[TOKEN:" + token + "]"
        segments.append(substitute)
        cursor = end
    segments.append(text[cursor:])
    return {**base, "status": "processed", "output": "".join(segments)}
'''
print("Lambda source ready:", len(LAMBDA_SOURCE.splitlines()), "lines")


**Teaching note:** In the tokenize action, the random token identifies a separately encrypted vault value. Reversing it would require a restricted service with permission to read the vault object and decrypt it with KMS. The notebook does not grant that access. `encrypt` protects the entire text and returns ciphertext. The two actions serve different application needs.


## Step 2C: deploy and invoke the Lambda function

Run the next cell only after the configuration prints your intended account and Region. Creating the function may take a few seconds. The notebook principal needs to pass the execution role to Lambda. If you use an existing role just created moments ago, IAM propagation may briefly delay deployment.


In [ ]:
from io import BytesIO
from zipfile import ZipFile, ZIP_DEFLATED
from uuid import uuid4

FUNCTION_NAME = None
if RUN_AWS:
    FUNCTION_NAME = "pii-demo-" + uuid4().hex[:12]
    zip_buffer = BytesIO()
    with ZipFile(zip_buffer, "w", ZIP_DEFLATED) as archive:
        archive.writestr("lambda_function.py", LAMBDA_SOURCE)
    lambda_client.create_function(
        FunctionName=FUNCTION_NAME, Runtime="python3.12", Role=LAMBDA_ROLE_ARN,
        Handler="lambda_function.lambda_handler", Code={"ZipFile": zip_buffer.getvalue()},
        Timeout=30, MemorySize=256,
        Environment={"Variables": {"VAULT_BUCKET": VAULT_BUCKET, "KMS_KEY_ARN": KMS_KEY_ARN}},
        Description="Slide 12 synthetic PII privacy demo; created from VS Code notebook",
    )
    lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION_NAME,
                                                         WaiterConfig={"Delay": 2, "MaxAttempts": 30})
    print("Lambda ready:", FUNCTION_NAME)
else:
    print("AWS is disabled; no function was deployed.")


**Comprehend check:** The next cell confirms which PII categories the service detected for this synthetic sentence. Models can vary by text and Region, so inspect this result before explaining the policy outputs.


In [ ]:
def invoke_privacy_action(action, text=SAMPLE_TEXT):
    if not RUN_AWS or not FUNCTION_NAME:
        raise RuntimeError("Enable RUN_AWS and run the deployment cell first.")
    response = lambda_client.invoke(
        FunctionName=FUNCTION_NAME, InvocationType="RequestResponse",
        Payload=json.dumps({"action": action, "text": text}).encode("utf-8"),
    )
    result = json.loads(response["Payload"].read())
    if response.get("FunctionError"):
        raise RuntimeError(f"Lambda failed on {action}: {result.get('errorType', '')} {result.get('errorMessage', '')}")
    return result

if RUN_AWS:
    detection = invoke_privacy_action("detect")
    print(json.dumps(detection, indent=2))
else:
    print("Run after AWS deployment to view actual Comprehend detections.")


### A. Mask and B. Replace

**Say:** “Mask preserves approximate length. Replace preserves the type of sensitive field.” The Lambda keeps non-PII text untouched. Detection can miss PII or mark extra spans; demonstrate this by looking at the actual result, and avoid assuming either transformation makes text safe without testing.


In [ ]:
if RUN_AWS:
    for action in ("mask", "replace"):
        result = invoke_privacy_action(action)
        print("ACTION:", action.upper(), "| STATUS:", result["status"])
        print(result.get("output", result.get("message")))
else:
    print("Enable AWS to run the real mask and replace actions.")


### C. Tokenize and D. Encrypt

**Say:** “A token lets an approved process look up the protected value later. Encryption returns ciphertext that only a principal with the right key permissions can decrypt.” Token-vault objects are written under `pii-demo/vault/`. Encryption returns base64 **encoding of KMS ciphertext**, not base64 of the original text.


In [ ]:
if RUN_AWS:
    tokenized = invoke_privacy_action("tokenize")
    encrypted = invoke_privacy_action("encrypt")
    print("TOKENIZE |", tokenized["status"], "|", tokenized.get("output", tokenized.get("message")))
    print("ENCRYPT  |", encrypted["status"], "| ciphertext bytes:",
          len(encrypted.get("ciphertext_base64", "")), "base64 characters")
    print("Ciphertext preview:", encrypted.get("ciphertext_base64", "")[:72] + "...")
else:
    print("Enable AWS to run the real tokenize and KMS encrypt actions.")


### E. Quarantine and F. Reject

**Say:** “Quarantine retains the record for controlled review. Reject returns a decision and does not persist the input in this demo.” The notebook prints only the S3 reference for quarantine; the sensitive sample itself has already been shown in step 1.


In [ ]:
if RUN_AWS:
    for action in ("quarantine", "reject"):
        result = invoke_privacy_action(action)
        print(action.upper(), json.dumps(result, indent=2))
else:
    print("Enable AWS to run the real quarantine and reject actions.")


## Debrief

Ask learners which action to use for each scenario:

- An analytics team needs aggregate sentiment but no identity: **mask or replace** before analysis.
- A service team needs to reconnect a case to a customer later: **tokenize**, with a separately protected vault.
- A restricted human reviewer needs to inspect a suspected privacy incident: **quarantine**.
- An application has no valid reason to accept sensitive input: **reject**.
- A system must preserve confidential original text while transmitting or storing it under key controls: **encrypt**.

**Limitations to teach:** Comprehend detects PII and can miss sensitive details. Add domain-specific validation and test false negatives. Only detected spans are transformed by mask, replace, and tokenize. Restrict access to notebook output and the S3 vault/quarantine prefixes. Do not copy real customer data into the sample cell or logs. KMS costs, Lambda calls, Comprehend analysis, and S3 storage can incur charges.

**Sources:** [Comprehend PII detection](https://docs.aws.amazon.com/comprehend/latest/dg/how-pii.html), [DetectPiiEntities API](https://docs.aws.amazon.com/boto3/latest/reference/services/comprehend/client/detect_pii_entities.html), [KMS Encrypt](https://docs.aws.amazon.com/boto3/latest/reference/services/kms/client/encrypt.html), [Lambda CreateFunction](https://docs.aws.amazon.com/boto3/latest/reference/services/lambda/client/create_function.html).


## Clean up the demo Lambda

This removes **only** the function created in this notebook. It leaves the user-managed IAM role, bucket, KMS key, and retained vault/quarantine objects in place for controlled review. Delete training objects according to your retention policy.


In [ ]:
if RUN_AWS and FUNCTION_NAME:
    lambda_client.delete_function(FunctionName=FUNCTION_NAME)
    print("Deleted Lambda function:", FUNCTION_NAME)
    FUNCTION_NAME = None
else:
    print("No demo Lambda function to delete.")
